# Hour 3 · Divination as decision trees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/3c_divination_trees.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F3c_divination_trees.ipynb)


**Goal:** formalize an omen text's *if → then* logic as data and visualize it as a tree — then discuss where LLMs help and where they mislead.

We use a real Ugaritic **birth-omen** text and a hand-built decision tree of it (`data/omens/`).

**Needs:** `networkx`.

*Reading:* `docs/07-divination.md`.

## 0. Setup


In [30]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_omen_text, load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

[loader] Loaded 278 CUC tablets, 25477 word tokens (source: AlexWalhai/cuc JSONL cache, licence: CC BY-NC 4.0).
Loaded 278 tablets — genre of the first one: myth


## 1. The omen text (English working translation)

The sample is Dennis Pardee's translation of **RS 24.247+** (KTU 1.103+1.145), the Ugaritic *Malformed Animal Fetuses* omens (Pardee, *Ritual and Cult at Ugarit*, 2002). Note the conditional pattern: *if the newborn lacks or shows feature X → consequence for land, king, or cattle*.

In [31]:
txt = load_omen_text()
print(txt[:900])


 As for the ewes of the sheep/goats, [when t]hey give birth:

 If it is a stone, many will fall in the land. 
 
 If it is a piece of wood, behold [ ]ôYû <ATR YLD, its cattle will be destroyed. 
 If the fetus is smooth, without h[air?], there will be []...in the land. 
 And if i[t has no ], the land will perish. 
 [ ] there will be famine in the land. 
 [ ] nor nostrils, the land [ ;] ditto. 
 [And] if it has no [ ], the king will sieze the lan[d of his enemy and?] the weapon of the king will lay the land low. 
 [ ] [ ] cattle [ will peri]sh.? 
 And if it has no [left] thigh, the king will [ ] his enemy. 
 And if there is no lower left leg, the king [will ] his enemy. 
 And if there is a horn of flesh [in] its lef[t te]mple, [ ]. 
 If it has no spleen [ ] [ ;] di[tto;] 
 the king will not obtain offspring. 
 [And] if it has no testicles, the (seed-)gra[in ].5 
 And if the middle part of 


## 2. The same logic, formalized as a tree (nested JSON)

In [32]:
from data.loader import load_omen_tree
import json
tree = load_omen_tree()
print(json.dumps(tree, indent=2, ensure_ascii=False)[:700], "...")

{
  "omen": {
    "sheep birth": {
      "missing body part": {
        "[body part] 1": "the land will perish",
        "[body part] 2": "the king will sieze the lan[d of his enemy and?] the weapon of the king will lay the land low",
        "nostrils": "[...]",
        "[...] and nostrils": "the land [...]; there will be famine in the land",
        "ḤRṢP in [its?] K[...]": "[...]",
        "thigh": {
          "left": "the king will [...] his enemy",
          "right": "[...]"
        },
        "leg": {
          "lower left leg": "the king will [...] his enemy",
          "(rear?) legs": "huradu troops will turn against the king",
          "left (fore?)leg": "the land of the enemy will ...


## 3. Turn the nested dict into a graph

In [33]:
import networkx as nx
G = nx.DiGraph()
counter = [0]
def add(node_label, subtree, parent=None):
    nid = counter[0]; counter[0]+=1
    G.add_node(nid, label=str(node_label), leaf=not isinstance(subtree, dict))
    if parent is not None: G.add_edge(parent, nid)
    if isinstance(subtree, dict):
        for k, v in subtree.items():
            add(k, v, nid)
    else:
        lid = counter[0]; counter[0]+=1
        G.add_node(lid, label="→ "+str(subtree)[:40], leaf=True)
        G.add_edge(nid, lid)
# Draw ONLY the sheep-birth omens here; the (separate) lunar tablet (RIH 78/14) is in §6.
add("sheep birth (RS 24.247+)", tree["omen"]["sheep birth"])
print("nodes:", G.number_of_nodes())

nodes: 77


## 4. Draw the sheep-birth decision tree

This is the **sheep-birth** tablet — **RS 24.247+** (KTU 1.103+1.145), Pardee's *Malformed Animal Fetuses*. Its data file also carries a short set of **lunar** omens, but those are a *different* tablet (RIH 78/14) — we draw them on their own in §6, not mixed into this tree.

In [34]:
import collections
import textwrap
import plotly.graph_objects as go

def layered_pos(G, root=0):
    depth = {root: 0}
    q = collections.deque([root])
    while q:
        u = q.popleft()
        for v in G.successors(u):
            if v not in depth:
                depth[v] = depth[u] + 1
                q.append(v)
    rows = collections.defaultdict(list)
    for n, d in depth.items():
        rows[d].append(n)
    pos = {}
    for d, ns in rows.items():
        for i, n in enumerate(ns):
            pos[n] = (d, -i)
    return pos

def wrap_label(text, width=34):
    return "<br>".join(textwrap.wrap(text, width=width))

def plot_interactive_tree(G, title, root=0):
    pos = layered_pos(G, root=root)
    labels = {n: G.nodes[n]["label"] for n in G.nodes()}
    colors = ["#f8dfaa" if not G.nodes[n]["leaf"] else "#dff5df" for n in G.nodes()]

    edge_x, edge_y = [], []
    for u, v in G.edges():
        xu, yu = pos[u]; xv, yv = pos[v]
        edge_x.extend([xu, xv, None]); edge_y.extend([yu, yv, None])

    node_x = [pos[n][0] for n in G.nodes()]
    node_y = [pos[n][1] for n in G.nodes()]
    node_text = [labels[n] for n in G.nodes()]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
        line=dict(color="rgba(120,120,120,0.55)", width=1.2),
        hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=node_x, y=node_y, mode="markers+text",
        text=[wrap_label(t) for t in node_text], textposition="middle center",
        textfont=dict(size=10, color="#111827"),
        marker=dict(size=22, color=colors, line=dict(color="#64748b", width=1)),
        hovertemplate="<b>%{hovertext}</b><extra></extra>", hovertext=node_text,
        showlegend=False))

    # Adaptive canvas: wider for deeper trees, taller for more stacked siblings,
    # so a big omen series stays legible instead of cramped.
    per_depth = collections.Counter(int(x) for x in node_x)
    width  = max(1000, (max(per_depth) + 1) * 330)
    height = max(600, max(per_depth.values()) * 26)
    fig.update_layout(
        title=title, width=width, height=height,
        margin=dict(l=20, r=20, t=60, b=20),
        paper_bgcolor="white", plot_bgcolor="white",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
                   range=[min(node_x) - 0.6, max(node_x) + 0.6]),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
                   range=[min(node_y) - 0.8, max(node_y) + 0.8]),
        dragmode="pan")
    fig.show()   # render once; do NOT `return fig` (that caused a duplicate display)

plot_interactive_tree(G, "Sheep-birth omen series — RS 24.247+ (KTU 1.103+1.145)")

## 5. Birth omens across two worlds — teratomancy

Birth-anomaly omens (*teratomancy*) were a shared Near-Eastern science. The Ugaritic and Babylonian versions are **two branches of one genre**, not copies of each other: Pardee stresses that the Ugaritic texts correspond to *no* known Mesopotamian tablet and read more like a *reasoned overview* than the sprawling Babylonian compendia.

Strikingly, **both** cultures split birth omens into **animal** and **human**:

| | Ugaritic (Pardee) | Babylonian (George) |
|---|---|---|
| **animal** births | RS 24.247+ — the §4 sheep tree | no. 12 — *miscarried foetus* |
| **human** births | RS 24.302 — a tiny fragment\* | no. 35 — *Šumma izbu* Tablet I |

We can tree three of these four; the Ugaritic human text is too broken. Same grammar throughout (**protasis → apodosis**), same fixation on the fate of **king and land**.

> *šumma sinništu nēša ūlid* → "If a woman gives birth to a lion: that town will be taken, its king will be captured." — *Šumma izbu* I 5 *(transliteration normalized from George 2013)*

\* RS 24.302 (KTU 1.140) preserves little beyond the opening *"when a woman gives birth …"* — but its very existence shows Ugarit, like Babylon, distinguished human from animal teratomancy.

In [35]:
# Pick up loader.py edits without a kernel restart (avoids a stale-import error).
import importlib, data.loader
importlib.reload(data.loader)
from data.loader import load_babylonian_izbu_tree, load_babylonian_foetus_tree

bab  = load_babylonian_izbu_tree()      # No. 35 — Šumma izbu I (human births)
foet = load_babylonian_foetus_tree()    # No. 12 — Old Babylonian miscarried-foetus omens

# Reusable: nested dict → graph (same recipe as the Ugaritic tree in cells 3–4).
def build_omen_graph(subtree, root_label):
    g = nx.DiGraph(); ctr = [0]
    def add(node_label, st, parent=None):
        nid = ctr[0]; ctr[0] += 1
        g.add_node(nid, label=str(node_label), leaf=not isinstance(st, dict))
        if parent is not None:
            g.add_edge(parent, nid)
        if isinstance(st, dict):
            for k, v in st.items():
                add(k, v, nid)
        else:
            lid = ctr[0]; ctr[0] += 1
            g.add_node(lid, label="→ " + str(st)[:40], leaf=True)
            g.add_edge(nid, lid)
    add(root_label, subtree)
    return g

print("No. 12 — miscarried-foetus omens, read by body feature:")
for feat, apo in foet["omen"]["horns"].items():
    print(f"  horns: {feat:24s} → {apo}")

print("\nNo. 35 — Šumma izbu I, human-birth omens:")
for who, apo in list(bab["omen"]["gives birth to an animal"].items())[:5]:
    print(f"  gives birth to a {who.split(' (')[0]:9s} → {apo}")

No. 12 — miscarried-foetus omens, read by body feature:
  horns: two on right, one on left → symbol of Sargon, who had no equal
  horns: two on left, one on right → enemy will hem you in and take away your territory
  horns: protruding               → there will be an all-powerful king in the land; alternative: the dynasty of the king and his sons is at an end

No. 35 — Šumma izbu I, human-birth omens:
  gives birth to a lion      → that town will be taken, its king will be captured
  gives birth to a wolf      → the people's minds will become deranged
  gives birth to a dog       → the head of the household will die and that household will be scattered; the people's minds will become deranged; plague will devour
  gives birth to a pig       → a woman will occupy the throne
  gives birth to a ox        → there will be an all-powerful king in the land


In [36]:
# No. 12 — the miscarried-foetus tree: the DIRECT parallel to the Ugaritic sheep omens
Gf = build_omen_graph(foet["omen"], "miscarried foetus")
plot_interactive_tree(Gf, "Babylonian teratomancy — miscarried-foetus omens (George no. 12)")

In [37]:
# No. 35 — the vivid Šumma izbu Tablet I human-birth tree
Gb = build_omen_graph(bab["omen"], "Šumma izbu I")
plot_interactive_tree(Gb, "Babylonian Šumma izbu Tablet I — human-birth omens (George no. 35)")

### Three birth-omen trees, two traditions

|  | Bab. *miscarried foetus* (no. 12) | Bab. *Šumma izbu* I (no. 35) | Ugaritic **RS 24.247+** |
|---|---|---|---|
| Birth of… | a malformed **animal** foetus | a **human** baby | a **sheep / goat** |
| Read by… | **body feature** (horns, eyes, hooves) | **species / shape** of the newborn | missing or odd **body part** |
| Form | protasis → apodosis | protasis → apodosis | protasis → apodosis |
| Outcome | king, land, dynasty, famine | king, throne, land, dynasty | king, land, cattle, enemy |
| Date | Old Babylonian (early 2nd mill.) | Standard Babylonian (1st mill.) | Late Bronze Age Ugarit |
| Source | George 2013, no. 12 | George 2013, no. 35 | Pardee 2002, text 42 |

The **foetus** omens (no. 12) are the closest kin to the Ugaritic text — both inspect a deformed **animal** newborn, feature by feature. The **human** *Šumma izbu* I (no. 35) is the same tradition's vivid first chapter, and Ugarit has its own (fragmentary) human counterpart (RS 24.302). All three trees run the identical *if → then* engine, fixated on the fate of king and land.

But recall the caution (`docs/07-divination.md`): the Babylonian series are **systematic** but not **empirical** (George — entries by analogy and wordplay, even impossible births), while the Ugaritic texts read as a **reasoned overview of the possibilities** (Pardee). The trees capture the *structure* of the reasoning, not a log of things observed — exactly the nuance an LLM will smooth over.

## 6. The same logic in the sky — celestial omens

Birth omens are only one branch. Ugarit also read the **heavens**, in a *separate* tablet — **RIH 78/14** (KTU 1.163), Pardee's *Lunar Omens* (the colour and newness of the moon, a falling star). Babylon did the same on a far grander scale: *Enūma Anu Enlil*, in which a **lunar eclipse** is decoded by its **watch**, the **direction** it clears, and above all the **calendar date** — every month, every key day, its own prediction. We draw the small Ugaritic lunar tablet as its own tree, then the Babylonian eclipse calendar as a **grid** (its natural shape):

In [38]:
import pandas as pd
from data.loader import load_babylonian_celestial_tree

# Ugaritic lunar omens — RIH 78/14, a SEPARATE tablet, drawn as its own little tree
ug_cel = load_omen_tree()["omen"]["celestial"]
Gc = build_omen_graph(ug_cel, "Ugaritic lunar omens (RIH 78/14)")
plot_interactive_tree(Gc, "Ugaritic lunar omens — RIH 78/14 (a separate tablet)")

# Babylonian lunar-eclipse — the 'general' rules, then the systematic month × day grid
cel = load_babylonian_celestial_tree()
ecl = cel["omen"]["celestial omens"]["lunar eclipse"]
print("Babylonian lunar-eclipse, general rules (a sample):")
for k, v in list(ecl["general"].items())[:4]:
    print(f"  eclipse {k:28s} → {v}")

grid = pd.DataFrame(ecl["month-specific"]).T
grid = grid.reindex(columns=sorted(grid.columns, key=lambda c: int(c.split()[1]))).fillna("—")
print(f"\nBabylonian eclipse calendar: {grid.shape[0]} months × {grid.shape[1]} key days "
      f"— every cell is its own omen:")
grid

Babylonian lunar-eclipse, general rules (a sample):
  eclipse evening watch                → signifies a fatal epidemic
  eclipse middle watch                 → signifies the decline of commerce
  eclipse morning watch                → signifies the end of a dynasty (or recovery from sickness)
  eclipse right-hand side reversed     → deluge will occur everywhere

Babylonian eclipse calendar: 11 months × 6 key days — every cell is its own omen:


,day 14,day 15,day 16,day 19,day 20,day 21
Nisannu,downfall of the king of Akkad; city and people...,famine; people will trade their infant childre...,—,redness of moon is orange-hued; abundance for ...,king will declare war on king,commerce will dwindle
Ayyaru,all-powerful king will die; land will perish,king and people are safe; war declared; shortf...,king of Amurru will die; his people will perish,—,—,—
Simanu,famous king will die; a prince will seize the ...,king of Amurru’s servants will kill him in revolt,king will fall in battle; watercourse will dry up,—,—,—
Dumuzi,inundation reduces barley; large people seek r...,rains removed; starvation,evil in the land; bounty lost,—,—,—
Abu,famine; king of Amurru will die,Nergal will devour; fatalities in the land,ewes will not bring lambs to term,—,—,—
Elul,king’s brother will seize throne in a revolt,king’s son will kill father and seize the throne,king of the upper land will enter deserted land,—,—,—
Tashritu,attack by Elam on my land; agriculture will th...,locust swarm will arise in spring and strike c...,king will die; dynasty over; land will be at war,—,—,—
Kislimu,attack by Gutian army; people will perish,enemy will demolish fortifications,king’s auxiliaries and troops will turn hostil...,—,fog and constant rumbles of thunder,civil war
Tebetu,"Adad will devastate men, livestock, and wild a...",rain removed; land’s crops lost,king without claim will seize throne; war,—,a people that experienced deportation will ret...,king of a lost people will return home
Addaru,dogs will go mad and bite people,lions will rampage and cut off traffic,an undefeated land will be defeated,—,defeat of Ur,Gutian army will conquer the land of Akkad


The Ugaritic sky omens are far **smaller** than the Babylonian system — a few lines beside a calendar that assigns a fate to an eclipse on *every* key day of *every* month. That captures George's point about the Babylonian series: "the outcome of cumulative attempts to embrace the entire universe in a system of reciprocal inferences," a would-be **theory of everything**. Yet Pardee cautions that the Ugaritic text is not a Babylonian *import* but an independent **Western** version of the genre. And birth + sky are not all: Ugarit's divinatory repertoire runs to **four** manuals (Pardee texts 42–45) — animal *and* human teratomancy, lunar omens, and **dream** omens (RS 18.041). See `docs/07`.

## 7. Where LLMs help — and where they are dangerous
**Help:** extract this structure from raw text, validate the JSON, **compare omen trees across cultures** (as we just did), generate a plain-language explanation.

**Danger:** *invent* missing branches (the texts are full of `[...]` gaps), *smooth over* ambiguity, *lose* philological detail.

> **Try it (presenter):** paste an omen text into an LLM and ask for this JSON tree — then check what it filled in that the tablet does **not** say. That gap is the lesson: philology + data modelling + LLM, with a human in the loop.